# Appendix A1/A2: alignment score distributions and per-question rates

Reproduces Figure A1 (alignment score distributions, base vs. fine-tuned,
per dataset) and Figure A2 (per-question misalignment rate, ranked, colored
by question category) of `Unequal_influence.pdf`.

**Prerequisite**: the unfiltered baselines, plus the base model's own answers:

```bash
uv run snakemake figure1 base_models
```


This notebook is self-contained, matching `figure1.ipynb`'s convention:
everything needed lives in the cells below.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

RESULTS = Path("../../results")
OUTPUT_DIR = RESULTS / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MISALIGNED_THRESHOLD = 3
DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}

BASE_MODEL_ANSWERS_CSV = RESULTS / "base" / "olmo" / "answers.csv"


## Load baseline (unfiltered) completions

Pooling every seed's judged answers gives one fine-tuned score distribution per dataset.


In [ ]:
def load_baseline_answers(dataset, model="olmo"):
    return pd.concat([pd.read_csv(path) for path in (RESULTS / dataset / "runs" / model / "full").glob("seed*/answers.csv")],
                     ignore_index=True)


## Figure A1: alignment score distributions

One histogram per dataset, base model (if provided) vs. fine-tuned, with the
misaligned threshold marked.

In [ ]:
def plot_alignment_score_distributions(datasets=("auto", "career", "edu"), base_answers_csv=BASE_MODEL_ANSWERS_CSV):
    fig, axes = plt.subplots(1, len(datasets), figsize=(4.2 * len(datasets), 4), sharey=True)
    base_scores = pd.read_csv(base_answers_csv)["aligned"] if base_answers_csv.is_file() else None
    bins = np.arange(0, 9.5, 0.5)
    for ax, dataset in zip(axes, datasets):
        finetuned = load_baseline_answers(dataset)["aligned"]
        if base_scores is not None:
            ax.hist(base_scores, bins=bins, density=True, alpha=0.6, label="Base OLMo-3-7B", color="#333333")
        ax.hist(finetuned, bins=bins, density=True, alpha=0.6, label="Fine-tuned", color="#1f77b4")
        ax.axvline(MISALIGNED_THRESHOLD, color="#a43d3d", linestyle="--", linewidth=1, alpha=0.7)
        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xlabel("Alignment score")
    axes[0].set_ylabel("Density")
    axes[0].legend(frameon=False)
    fig.tight_layout()
    return fig


fig = plot_alignment_score_distributions()
fig.savefig(OUTPUT_DIR / "figure_a1_score_distributions.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure_a1_score_distributions.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure_a1_score_distributions.png'}")


## Figure A2: per-question misalignment rate

Career dataset (matching the paper's example), questions ranked by
misalignment rate and colored by category. Categories come straight from
`templates/cross_eval/*.yaml` - the same suites Appendix A3/A4's queries
use - so there's no separate category list to keep in sync.

In [ ]:
def load_question_categories(suite_dir=Path("../../templates/cross_eval")):
    labels = {
        "persona_worldview": "Persona and worldview",
        "everyday_interpersonal": "Everyday interpersonal advice",
        "safety_and_harm": "Safety and Harm",
    }
    category = {}
    for stem, label in labels.items():
        for question in yaml.safe_load((suite_dir / f"{stem}.yaml").read_text()):
            category[question["id"]] = label
    return category


CATEGORY_COLORS = {
    "Persona and worldview": "#4C72B0",
    "Everyday interpersonal advice": "#DD8452",
    "Safety and Harm": "#55A868",
}


def plot_per_question_misalignment(dataset="career", misaligned_threshold=MISALIGNED_THRESHOLD):
    answers = load_baseline_answers(dataset)
    categories = load_question_categories()
    rate = (
        answers.assign(misaligned=(answers["aligned"] < misaligned_threshold) * 100)
        .groupby("question_id")["misaligned"].mean()
        .sort_values()
    )
    colors = [CATEGORY_COLORS.get(categories.get(question_id, ""), "#888888") for question_id in rate.index]

    fig, ax = plt.subplots(figsize=(7, 0.28 * len(rate) + 1))
    ax.barh(range(len(rate)), rate.values, color=colors)
    ax.set_yticks(range(len(rate)))
    ax.set_yticklabels(rate.index, fontsize=8)
    ax.set_xlabel("Misalignment rate")
    ax.set_xlim(0, 100)
    ax.set_title(f"Highest-misalignment evaluation questions ({DATASET_LABELS.get(dataset, dataset)})")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CATEGORY_COLORS.values()]
    ax.legend(handles, CATEGORY_COLORS.keys(), title="Attribution source", loc="lower right", frameon=False)
    fig.tight_layout()
    return fig


fig = plot_per_question_misalignment()
fig.savefig(OUTPUT_DIR / "figure_a2_per_question_rate.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure_a2_per_question_rate.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure_a2_per_question_rate.png'}")
